# 1.1 Pipeline Deployment Model Sentimen Tokopedia
Pipeline yang dibuat meliputi: 1. Persiapan environment dan konfigurasi kredensial Kaggle. 2. Unduhan dataset mentah. 3. Pra-pemrosesan dan pelabelan data ulasan. 4. Pelatihan model Machine Learning dengan Pipeline Scikit-Learn. 5. Serialisasi objek model ke format Pickle. 6. Pembuatan antarmuka pengguna dengan Streamlit. 7. Deployment publik menggunakan Localtunnel. 8. Interpretasi teoretis konsep MLOps.

# 1.2 Tahap 1 Persiapan Lingkungan & Konfigurasi Kaggle API
Tahap ini diawali dengan instalasi pustaka yang dibutuhkan untuk analitik teks dan deployment antarmuka, serta mengatur otentikasi Kaggle agar pengunduhan dataset dapat dilakukan secara terprogram.

In [1]:
import subprocess
import sys
import os
import shutil
from pathlib import Path

# Install required packages locally
ml_packages = ["pandas", "scikit-learn", "kaggle", "streamlit", "Sastrawi"]
print("Installing required packages...")
for pkg in ml_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages installed successfully.")

# Configure Kaggle API for local/Colab environment
home_path = Path.home()
kaggle_config_dir = home_path / ".kaggle"
kaggle_config_dir.mkdir(exist_ok=True)

kaggle_token_source = Path("kaggle.json")
if kaggle_token_source.exists():
    kaggle_token_dest = kaggle_config_dir / "kaggle.json"
    shutil.copy(kaggle_token_source, kaggle_token_dest)

    if os.name != 'nt': # Set permission on Unix/Linux
        os.chmod(kaggle_token_dest, 0o600)
    print(f"Kaggle API token configured at: {kaggle_token_dest}")
else:
    raise FileNotFoundError("File kaggle.json tidak ditemukan. Pastikan Anda telah mengunggahnya ke direktori kerja.")

Installing required packages...
All packages installed successfully.
Kaggle API token configured at: C:\Users\abdul\.kaggle\kaggle.json


# 1.2.1 Download dan Ekstrak Dataset dari Kaggle
Dataset ulasan Tokopedia diunduh langsung melalui terminal yang dieksekusi dari Python menggunakan dataset ID yang dispesifikasikan: salmanabdu/tokopedia-product-reviews-2025.

In [2]:
dataset_id = "salmanabdu/tokopedia-product-reviews-2025"
data_folder = Path("./dataset_tokopedia")
data_folder.mkdir(parents=True, exist_ok=True)

print("Downloading dataset from Kaggle...")
subprocess.run(
    ["kaggle", "datasets", "download", "-d", dataset_id, "-p", str(data_folder), "--force"],
    check=True
)

zip_filepath = data_folder / "tokopedia-product-reviews-2025.zip"
if zip_filepath.exists():
    print(f"Extracting {zip_filepath}...")
    shutil.unpack_archive(zip_filepath, data_folder)
    print("Extraction completed.")
else:
    raise FileNotFoundError("File ZIP dataset tidak ditemukan setelah proses download.")

Extracting dataset_tokopedia\tokopedia-product-reviews-2025.zip...
Extraction completed.


# 1.2.2 Interpretasi Teknis 1
Sama seperti pendekatan pemrosesan Big Data, pengelolaan environment untuk Machine Learning harus dipastikan reproducible. Konfigurasi API Kaggle mempermudah pengambilan data secara konsisten tanpa pengunduhan manual. Pustaka inti yang diinisiasi adalah pandas untuk manipulasi data tabular, scikit-learn untuk pemodelan ML klasik, dan streamlit untuk pembuatan dashboard interaktif secara pesat (RAD).

# 1.3 Pra-pemrosesan Data & Sampling
Berdasarkan instruksi, data perlu dibersihkan dari nilai kosong (null), dilabeli menjadi kategori "Positif", "Negatif", atau "Netral" berdasarkan nilai Rating, lalu diambil sampelnya agar pelatihan berjalan efisien di komputasi cloud gratis.

In [3]:
import pandas as pd
import glob

# Load the extracted CSV
csv_files = glob.glob(f"{str(data_folder)}/**/*.csv", recursive=True)
df_raw = pd.read_csv(csv_files[0])

# 1. Membersihkan nilai kosong
df_clean = df_raw.dropna(subset=['review_text', 'rating']).copy()

# 2. Membuat fitur target 'Sentimen'
def assign_sentiment(score):
    if score > 3: return 'Positif'
    elif score < 3: return 'Negatif'
    else: return 'Netral'

df_clean['Sentimen'] = df_clean['rating'].apply(assign_sentiment)

# 3. Preprocessing teks (Bahasa Indonesia) dan stemming bila tersedia
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer = StemmerFactory().create_stemmer()
except Exception:
    stemmer = None

import re

def clean_text(s):
    s = str(s).lower().strip()
    s = re.sub(r'http\S+', ' ', s)
    s = re.sub(r'[^0-9a-z\s]', ' ', s)
    s = re.sub(r'\b(ga|gak|nggak|enggak|tdk)\b', ' tidak ', s)
    s = re.sub(r'(.)\1{2,}', r'\1\1', s)
    s = re.sub(r'\s+', ' ', s).strip()
    if stemmer:
        s = stemmer.stem(s)
    return s

# Terapkan preprocessing
df_clean['clean'] = df_clean['review_text'].apply(clean_text)

# 4. Pengambilan sampel data dengan guard
n = 10000
n = min(n, len(df_clean))
df_model_ready = df_clean.sample(n=n, random_state=17)

print(f"Total data sampel untuk pelatihan: {len(df_model_ready)} baris")
display(df_model_ready[['review_text', 'clean', 'rating', 'Sentimen']].head())

Total data sampel untuk pelatihan: 10000 baris


,review_text,clean,rating,Sentimen
40190,packing sangat rapi dan aman terlihat dipackin...,packing sangat rapi dan aman lihat dipacking d...,5,Positif
60530,Terbaik!.................,baik,5,Positif
53758,seller gercep..packing rapih..barang oke sesua...,seller gercep packing rapih barang oke sesuai ...,5,Positif
21166,dikemas dengan baik dan di warp dengan plastik...,kemas dengan baik dan di warp dengan plastik w...,5,Positif
3430,Langganan beli di sini - daging nya selalu seg...,langgan beli di sini daging nya selalu segar d...,5,Positif


# 1.4 Pelatihan & Serialisasi Model
Model klasifikasi yang digunakan adalah Logistic Regression dengan fitur gabungan TF-IDF word n-gram dan char n-gram. GridSearchCV digunakan untuk memilih hyperparameter terbaik berdasarkan `f1_macro` agar lebih adil pada kelas yang tidak seimbang. Model terbaik kemudian diserialisasi menggunakan modul pickle.

In [4]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
import pickle
from sklearn.metrics import classification_report, confusion_matrix
import warnings
from sklearn.exceptions import ConvergenceWarning
import pandas as pd

warnings.filterwarnings("ignore", category=ConvergenceWarning)

X_raw = df_model_ready['review_text']
X = df_model_ready['clean']
y = df_model_ready['Sentimen']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=17, stratify=y
)

word_tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.9, min_df=3, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5, sublinear_tf=True)
features = FeatureUnion([
    ('word', word_tfidf),
    ('char', char_tfidf)
])

pipe = Pipeline([
    ('features', features),
    ('clf', LogisticRegression(max_iter=1000, tol=1e-3, solver='saga', n_jobs=-1))
])

param_grid = {
    'clf__C': [0.5, 1, 5],
    'clf__class_weight': [None, 'balanced']
}

gs = GridSearchCV(pipe, param_grid, scoring='f1_macro', cv=5, n_jobs=-1, verbose=1)
print("Melatih model dengan GridSearchCV (scoring=f1_macro)...")
gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
best_model = gs.best_estimator_
y_pred = best_model.predict(X_test)
print("Classification report:")
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

# save model
output_model_file = "model_tokopedia.pkl"
with open(output_model_file, 'wb') as f:
    pickle.dump(best_model, f)
print(f"Objek model berhasil disimpan di: {output_model_file}")

# save misclassified examples
probs = None
if hasattr(best_model, 'predict_proba'):
    probs = best_model.predict_proba(X_test)

raw_test = X_raw.loc[X_test.index]
df_eval = pd.DataFrame({
    'text_raw': raw_test,
    'text_clean': X_test,
    'true': y_test,
    'pred': y_pred
})

if probs is not None:
    df_eval['max_prob'] = probs.max(axis=1)
    classes = best_model.named_steps['clf'].classes_
    for i, c in enumerate(classes):
        df_eval[f'prob_{c}'] = probs[:, i]

df_eval = df_eval.reset_index(drop=True)
misclassified = df_eval[df_eval['true'] != df_eval['pred']]
misclassified.to_csv('misclassified_examples.csv', index=False)
print(f"Jumlah misclassified pada test set: {len(misclassified)}")
display(misclassified.head(10))

Melatih model dengan GridSearchCV (scoring=f1_macro)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'clf__C': 5, 'clf__class_weight': None}
Classification report:
              precision    recall  f1-score   support

     Negatif       0.50      0.09      0.15        23
      Netral       0.38      0.10      0.16        29
     Positif       0.98      1.00      0.99      1948

    accuracy                           0.97      2000
   macro avg       0.62      0.40      0.43      2000
weighted avg       0.96      0.97      0.97      2000

Confusion matrix:
[[   2    3   18]
 [   1    3   25]
 [   1    2 1945]]
Objek model berhasil disimpan di: model_tokopedia.pkl
Jumlah misclassified pada test set: 50


,text_raw,text_clean,true,pred,max_prob,prob_Negatif,prob_Netral,prob_Positif
35,sayang banget 1 telurnya busuk dan 1 telur pecah.,sayang banget 1 telur busuk dan 1 telur pecah,Negatif,Positif,0.943796,0.016274,0.039930,0.943796
106,ternyata setang nya ga bisa dibelokin jadi lur...,nyata setang nya tidak bisa dibelokin jadi lur...,Positif,Netral,0.713319,0.139930,0.713319,0.146751
172,harga 3 juta tidak ada batrenya,harga 3 juta tidak ada batrenya,Negatif,Positif,0.961119,0.012625,0.026257,0.961119
247,yah ada tambahan aneh,yah ada tambah aneh,Netral,Positif,0.961790,0.009738,0.028472,0.961790
290,warna yg dikirim beda dgn yg dipesan. padahal ...,warna yg kirim beda dgn yg pes padahal belum s...,Netral,Negatif,0.764023,0.764023,0.135832,0.100145
310,biasanya langganan dan puas tp skg order 1500 ...,biasa langgan dan puas tp skg order 1500 gr yg...,Netral,Positif,0.963299,0.007297,0.029404,0.963299
319,ih sumpah ga suka bnget warna nya ga sesuai am...,ih sumpah tidak suka bnget warna nya tidak ses...,Negatif,Positif,0.878687,0.065747,0.055566,0.878687
369,"D luar expetasi,ternyata kecil bgt...kirain ha...",d luar expetasi nyata kecil bgt kirain harga 2...,Negatif,Positif,0.746381,0.151727,0.101893,0.746381
413,"barangnya sesuai, tapi ga bisa dibuka dratnya ...",barang sesuai tapi tidak bisa buka drat terlal...,Negatif,Positif,0.899718,0.068274,0.032007,0.899718
439,ukuran kekecilan dan penguncinya enggak baik,ukur kecil dan kunci tidak baik,Negatif,Positif,0.858831,0.101600,0.039569,0.858831


# 1.4.1 Interpretasi Teknis 2
Pipeline Scikit-Learn menggabungkan TF-IDF word n-gram dan char n-gram agar model lebih peka terhadap kata pendek atau variasi ejaan (contoh: "jelek", "jelekkk"). Pemilihan model menggunakan GridSearchCV dengan metrik `f1_macro` supaya performa pada kelas minoritas (Negatif/Netral) tidak tertutupi oleh dominasi kelas Positif. Hasil run terbaru menunjukkan akurasi dan weighted F1 tinggi karena kelas Positif sangat dominan, namun macro F1 masih lebih rendah. Ini menandakan performa pada kelas minoritas masih perlu ditingkatkan, misalnya melalui penyeimbangan data atau augmentasi pada kelas Negatif/Netral. File `misclassified_examples.csv` dapat digunakan untuk analisis error yang lebih terarah.

# 2.1 Pengembangan Dashboard & Tunneling
Agar model dapat diuji coba secara fungsional oleh pengguna akhir, antarmuka Streamlit diimplementasikan dan di-ekspos ke internet publik.

In [5]:
%%writefile app.py
import streamlit as st
import pickle
import os
import re
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer = StemmerFactory().create_stemmer()
except Exception:
    stemmer = None


def clean_text(s):
    s = str(s).lower().strip()
    s = re.sub(r'http\S+', ' ', s)
    s = re.sub(r'[^0-9a-z\s]', ' ', s)
    s = re.sub(r'\b(ga|gak|nggak|enggak|tdk)\b', ' tidak ', s)
    s = re.sub(r'(.)\1{2,}', r'\1\1', s)
    s = re.sub(r'\s+', ' ', s).strip()
    if stemmer:
        s = stemmer.stem(s)
    return s

@st.cache_resource
def load_sentiment_model():
    model_path = 'model_tokopedia.pkl'
    if os.path.exists(model_path):
        with open(model_path, 'rb') as f:
            return pickle.load(f)
    return None

model = load_sentiment_model()

st.set_page_config(page_title="Sentimen Tokopedia", layout="centered")
st.title("Analisis Sentimen Ulasan Tokopedia")
st.markdown("""
Aplikasi ini menggunakan model **Logistic Regression** dengan TF-IDF word + char n-gram untuk memprediksi kategori sentimen dari teks ulasan produk Tokopedia.
""")

st.divider()

user_review = st.text_area(
    "Masukkan ulasan pelanggan di bawah ini:",
    placeholder="Contoh: Barang bagus, pengiriman sangat cepat...",
    height=150
)

if st.button("Prediksi Sentimen"):
    if model is None:
        st.error("File 'model_tokopedia.pkl' tidak ditemukan. Pastikan serialisasi model sudah berhasil.")
    elif user_review.strip() == "":
        st.warning("Mohon masukkan teks ulasan terlebih dahulu untuk melakukan prediksi.")
    else:
        txt = clean_text(user_review)
        if hasattr(model, 'predict_proba'):
            probs = model.predict_proba([txt])[0]
            classes = model.named_steps['clf'].classes_
            top_idx = probs.argmax()
            prediction = classes[top_idx]
            confidence = probs[top_idx]

            st.subheader("Hasil Analisis:")
            if prediction == "Positif":
                st.success(f"Sentimen Terdeteksi: **{prediction}** — Konfidence: {confidence:.2f}")
            elif prediction == "Negatif":
                st.error(f"Sentimen Terdeteksi: **{prediction}** — Konfidence: {confidence:.2f}")
            else:
                st.info(f"Sentimen Terdeteksi: **{prediction}** — Konfidence: {confidence:.2f}")

            if confidence < 0.60:
                st.warning("Konfidence rendah, hasil bisa kurang stabil. Pertimbangkan evaluasi manual.")

            st.write("Probabilitas per kelas:")
            for c, p in zip(classes, probs):
                st.write(f"- {c}: {p:.2f}")
        else:
            prediction = model.predict([txt])[0]
            st.subheader("Hasil Analisis:")
            if prediction == "Positif":
                st.success(f"Sentimen Terdeteksi: **{prediction}**")
            elif prediction == "Negatif":
                st.error(f"Sentimen Terdeteksi: **{prediction}**")
            else:
                st.info(f"Sentimen Terdeteksi: **{prediction}**")

st.caption("Vinix7 Project-Based Internship - Kelompok 17")

Writing app.py


# 2.2 Interpretasi Teknis Tahap 2
Pengembangan dashboard ini bertujuan untuk mendemonstrasikan hasil pemodelan Machine Learning dalam bentuk yang dapat diuji secara mandiri oleh tim non-teknis (seperti tim marketing) sesuai permintaan klien. Berikut adalah beberapa poin teknis utama:  Serialisasi Model: Aplikasi menggunakan modul pickle untuk memuat file model_tokopedia.pkl. Penggunaan dekorator @st.cache_resource pada fungsi pemuatan model bertujuan untuk mengoptimalkan performa aplikasi agar model hanya dimuat satu kali ke memori selama sesi aplikasi berjalan.  Modularitas UI: Antarmuka dirancang dengan komponen st.text_area untuk menerima input teks ulasan yang fleksibel. Penggunaan st.button memicu proses inference secara real-time.  Visualisasi & Feedback: Hasil prediksi divisualisasikan menggunakan skema warna standar Streamlit (st.success/st.error/st.info) dan dilengkapi skor konfidence serta probabilitas per kelas agar pengguna memahami tingkat kepastian prediksi.  Error Handling: Skrip menyertakan pengecekan keberadaan file model dan validasi input kosong untuk mencegah aplikasi mengalami crash saat dijalankan di lingkungan produksi.

# 3.1 Tahap 3: Deployment & Tunneling

Pada tahap ini, aplikasi Streamlit yang telah dibuat diaktifkan di latar belakang sistem Google Colab. Karena Colab berjalan di infrastruktur cloud tertutup, digunakan layanan tunneling (Localtunnel) untuk menciptakan URL publik agar dasbor dapat diakses oleh pengguna luar atau tim marketing.

In [ ]:
# 1. Instalasi localtunnel melalui npm
!npm install -g localtunnel

# 2. Mengambil IP Publik sebagai password otentikasi localtunnel
import urllib.request
print("===============================================================")
print("KODE PASSWORD (IP) UNTUK AKSES URL:")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("===============================================================")

# 3. Menjalankan Streamlit di latar belakang dan membuka tunnel pada port 8501
# Output dialihkan ke logs.txt agar tidak memblokir sel eksekusi
get_ipython().system_raw('streamlit run app.py & npx localtunnel --port 8501 > logs.txt &')

print("Aplikasi sedang dipersiapkan di latar belakang.")
print("Silakan cek URL publik yang akan muncul di bawah atau di logs.txt.")


changed 22 packages in 1s

3 packages are looking for funding
  run `npm fund` for details
KODE PASSWORD (IP) UNTUK AKSES URL:
180.254.77.101


In [ ]:
import urllib.request
import time

print("===============================================================")
print("1. KODE PASSWORD (IP) ANDA ADALAH:")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("===============================================================")

# Jalankan Streamlit di latar belakang
get_ipython().system_raw('streamlit run app.py &')

# Beri waktu sedikit agar Streamlit menyala
time.sleep(3)

print("\n2. KLIK TAUTAN LOCALTUNNEL DI BAWAH INI:")
# Jalankan Localtunnel di foreground agar URL-nya langsung muncul
!npx localtunnel --port 8501

1. KODE PASSWORD (IP) ANDA ADALAH:
180.254.77.101


## 3.2 Bukti Deployment Dashboard

Setelah aplikasi Streamlit berhasil dijalankan melalui Localtunnel, dashboard dapat diakses menggunakan URL publik yang muncul pada output cell sebelumnya. URL tersebut digunakan untuk membuka aplikasi dari browser, kemudian dilakukan pengujian dengan memasukkan contoh ulasan pelanggan Tokopedia.

Contoh pengujian yang dapat digunakan:

"Barang bagus, pengiriman cepat, seller ramah dan produk sesuai deskripsi."

Jika dashboard berhasil menampilkan hasil prediksi sentimen, maka proses deployment sederhana telah berjalan dengan baik. Screenshot dashboard yang menampilkan judul aplikasi, kotak input ulasan, tombol prediksi, dan hasil prediksi perlu dilampirkan sebagai bukti bahwa aplikasi berhasil diakses secara publik.

## 3.3 Screenshot Dashboard (Bukti Uji Coba)

**1) Netral**

![Uji Netral](screenshots/netral.png)

Caption: Prediksi Netral pada ulasan komplain ringan, confidence terlihat di panel hasil, serta probabilitas per kelas ditampilkan.

**2) Negatif**

![Uji Negatif](screenshots/negatif.png)

Caption: Prediksi Negatif pada ulasan bernada keluhan keras, confidence dan distribusi probabilitas per kelas terlihat jelas.

**3) Positif**

![Uji Positif](screenshots/positif.png)

Caption: Prediksi Positif pada ulasan yang menyatakan kualitas dan pelayanan baik, confidence tinggi.

**4) Low Confidence**

![Uji Confidence Rendah](screenshots/not-confidence.png)

Caption: Contoh prediksi dengan confidence rendah sehingga muncul peringatan agar dilakukan evaluasi manual.

In [1]:
# Cell opsional untuk memastikan file penting sudah tersedia sebelum lanjut ke tahap Docker
import os

required_files = ["app.py", "model_tokopedia.pkl"]

for file in required_files:
    if os.path.exists(file):
        print(f"{file} ditemukan.")
    else:
        print(f"{file} belum ditemukan. Pastikan cell sebelumnya sudah dijalankan dengan benar.")

app.py ditemukan.
model_tokopedia.pkl ditemukan.


# 4. Tahap 4: Pemahaman MLOps & Containerization

Tahap ini menjelaskan bagaimana aplikasi sentimen Tokopedia dapat dikemas ke dalam container menggunakan Docker, serta bagaimana konsep MLOps digunakan untuk menjaga performa model setelah aplikasi berjalan di lingkungan produksi.

## 4.1 Dockerfile untuk Aplikasi Streamlit

Docker digunakan untuk mengemas aplikasi, model, dan dependensi ke dalam satu lingkungan yang konsisten. Dengan container, aplikasi Streamlit dapat dijalankan pada server lain tanpa perlu melakukan konfigurasi manual yang berbeda-beda. File utama yang dibutuhkan adalah `app.py`, `model_tokopedia.pkl`, dan `requirements.txt`.

In [2]:
%%writefile requirements.txt
streamlit
pandas
scikit-learn
numpy
Sastrawi

Writing requirements.txt


In [3]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY model_tokopedia.pkl .

EXPOSE 8501

CMD ["streamlit", "run", "app.py", "--server.address=0.0.0.0", "--server.port=8501"]

Writing Dockerfile


Dockerfile di atas dimulai dari image `python:3.11-slim` agar ukuran container lebih ringan. Direktori kerja aplikasi dibuat pada folder `/app`. Setelah itu, file `requirements.txt` disalin dan digunakan untuk memasang library yang diperlukan, yaitu Streamlit, Pandas, Scikit-Learn, NumPy, dan Sastrawi.

File `app.py` dan `model_tokopedia.pkl` kemudian disalin ke dalam container. Port 8501 dibuka karena Streamlit secara default berjalan pada port tersebut. Perintah terakhir menjalankan aplikasi Streamlit agar dapat diakses dari luar container.

Contoh perintah untuk membangun dan menjalankan container secara lokal:

```bash
docker build -t tokopedia-sentiment-app .
docker run -p 8501:8501 tokopedia-sentiment-app
```


## 4.2 Monitoring Data Drift

Jika akurasi prediksi model sentimen Tokopedia menurun drastis tahun depan padahal tidak ada kode yang diubah, fenomena tersebut dapat disebut sebagai data drift atau concept drift. Data drift terjadi ketika distribusi data baru berbeda dari data yang digunakan saat pelatihan model. Pada kasus ulasan Tokopedia, perubahan ini dapat muncul karena gaya bahasa pelanggan berubah, muncul istilah baru, tren produk berubah, atau pola keluhan pelanggan tidak lagi sama seperti data historis.

Concept drift terjadi ketika hubungan antara input dan label berubah. Misalnya, kata atau frasa yang sebelumnya dianggap netral dapat berubah menjadi indikasi sentimen negatif dalam konteks baru. Akibatnya, model yang sebelumnya akurat menjadi kurang relevan karena pola bahasa dan perilaku pelanggan sudah bergeser.

Dalam kerangka MLOps, solusi yang tepat adalah membangun sistem monitoring performa model secara berkelanjutan. Tim perlu memantau distribusi input ulasan, distribusi hasil prediksi, confidence score, jumlah prediksi tiap kelas, serta akurasi model jika label aktual tersedia. Jika terdeteksi penurunan performa, pipeline retraining perlu dijalankan menggunakan data terbaru yang sudah dibersihkan dan divalidasi.

Setelah model baru dilatih, performanya harus dibandingkan dengan model lama menggunakan metrik evaluasi seperti accuracy, precision, recall, dan F1-score. Model baru hanya boleh dipromosikan ke production jika hasilnya lebih stabil dan lebih baik daripada model sebelumnya. Dengan pendekatan ini, sistem dapat tetap relevan terhadap perubahan data pelanggan dari waktu ke waktu.

## 4.3 CI/CD untuk Penggantian Model Tanpa Downtime

CI/CD membantu tim mengganti versi model di server secara otomatis, terkontrol, dan minim risiko. CI atau Continuous Integration digunakan untuk memeriksa apakah kode aplikasi, file model, dan dependensi masih berjalan dengan benar. Misalnya, setiap kali ada model baru, pipeline CI dapat menjalankan pengecekan otomatis seperti validasi file `model_tokopedia.pkl`, pengujian apakah model dapat dimuat oleh aplikasi, dan pengujian apakah aplikasi Streamlit dapat menghasilkan prediksi.

CD atau Continuous Deployment digunakan untuk mengirim versi aplikasi atau model terbaru ke server production. Agar dashboard tidak mengalami downtime, proses deployment dapat menggunakan strategi blue-green deployment, rolling deployment, atau canary release. Pada blue-green deployment, versi lama tetap berjalan melayani pengguna, sementara versi baru dijalankan di lingkungan terpisah. Setelah versi baru lolos health check, trafik pengguna dialihkan ke versi baru.

Jika model baru mengalami error, trafik dapat dikembalikan ke versi lama sehingga pengguna tetap dapat mengakses dashboard. Dengan cara ini, tim dapat memperbarui model sentimen Tokopedia tanpa menghentikan layanan yang sedang berjalan. Pendekatan CI/CD juga membuat proses deployment lebih rapi karena setiap perubahan model terdokumentasi, dapat diuji, dan dapat di-rollback jika terjadi masalah.

## 4.4 Kesimpulan

Pipeline deployment model sentimen Tokopedia telah mencakup proses persiapan data, pelabelan sentimen berdasarkan rating, pelatihan model klasifikasi teks, serialisasi model ke dalam file `model_tokopedia.pkl`, pembuatan dashboard Streamlit, serta deployment sederhana menggunakan Localtunnel. Dari sisi MLOps, aplikasi ini dapat dikembangkan lebih lanjut dengan containerization menggunakan Docker, monitoring data drift, retraining model secara berkala, dan CI/CD untuk memperbarui model tanpa downtime.

Dengan alur tersebut, model Machine Learning tidak hanya berhenti pada tahap eksperimen, tetapi juga dapat dioperasikan dalam bentuk aplikasi interaktif yang dapat digunakan oleh tim bisnis untuk melakukan pengujian sentimen pelanggan secara mandiri.